# KV Cache Compression — 9 Methods Unified Comparison on LongBench

**Model:** Qwen/Qwen2.5-1.5B-Instruct  
**Tasks:** narrativeqa, qasper, multifieldqa_en, gov_report, hotpotqa, 2wikimqa  
**Budgets:** 10%, 20%, 50% of prefill length

| Method | Paper | Key idea |
|---|---|---|
| Baseline | — | Full KV cache, no compression |
| StreamingLLM | Xiao et al., 2023 | Attention sinks + FIFO sliding window |
| H2O | Zhang et al., NeurIPS 2023 | Cumulative attn → heavy-hitter top-K + recency |
| SnapKV | Li et al., 2024 | Pool prefill attention (observation window), compress once |
| KNorm | Devoto et al., EMNLP 2024 | Key L2-norm as importance, compress once |
| ScissorHands | Liu et al., 2024 | Persistent-token retention via cumul. attention |
| AdaKV | Fu et al., 2024 | Per-layer adaptive budget based on attn entropy |
| RKV | NeurIPS 2025 | Joint: attn importance × key non-redundancy |
| ChunkKV | NeurIPS 2025 | Chunk-level importance + layer-index reuse |

All methods use **Qwen2.5-1.5B-Instruct** with `attn_implementation="eager"` and share identical
prompt formatting, scoring, and evaluation loops for a fair comparison.

In [ ]:
!pip install -q transformers accelerate datasets sentencepiece \
             rouge-score matplotlib tqdm pandas huggingface_hub

In [ ]:
import gc, json, math, re, string, time, zipfile
from collections import Counter
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from rouge_score import rouge_scorer
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.cache_utils import DynamicCache

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID              = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE                = "cuda" if torch.cuda.is_available() else "cpu"

TASKS                 = ["narrativeqa","qasper","multifieldqa_en",
                          "gov_report","hotpotqa","2wikimqa"]
BUDGETS               = [0.5, 0.2, 0.1]      # keep-ratios (fraction of prefill kept)
MAX_EXAMPLES_PER_TASK = 50
MAX_INPUT_TOKENS      = 4096
MAX_NEW_TOKENS        = 64
RECENT_WINDOW         = 128   # tail tokens always pinned in every cache

METHODS = ["baseline","streamingllm","h2o","snapkv","knorm",
           "scissorhands","adakv","rkv","chunkkv"]

OUTPUT_DIR = Path("kv_unified_outputs")
DATA_DIR   = Path("longbench_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    attn_implementation="eager",   # required for output_attentions=True
    low_cpu_mem_usage=True,
)
if DEVICE == "cpu":
    model = model.to(DEVICE)
model.eval()
print(f"Loaded: {MODEL_ID}  ({model.config.num_hidden_layers} layers)")

In [ ]:
# ─── _OurCache ────────────────────────────────────────────────────────────────
# Subclass of DynamicCache that writes key_cache / value_cache into plain Python
# lists regardless of the transformers version (>= 4.47 redesigned the internals;
# Qwen2 also creates a HybridCache by default).

class _OurCache(DynamicCache):
    def __init__(self):
        try:
            super().__init__()
        except Exception:
            pass
        object.__setattr__(self, "key_cache", [])
        object.__setattr__(self, "value_cache", [])
        if not hasattr(self, "_seen_tokens"):
            object.__setattr__(self, "_seen_tokens", 0)

    def update(self, key_states, value_states, layer_idx, cache_kwargs=None):
        kc = object.__getattribute__(self, "key_cache")
        vc = object.__getattribute__(self, "value_cache")
        while len(kc) <= layer_idx:
            kc.append(None); vc.append(None)
        if kc[layer_idx] is None:
            kc[layer_idx] = key_states
            vc[layer_idx] = value_states
        else:
            kc[layer_idx] = torch.cat([kc[layer_idx], key_states], dim=-2)
            vc[layer_idx] = torch.cat([vc[layer_idx], value_states], dim=-2)
        try:
            object.__setattr__(self, "_seen_tokens", kc[0].shape[-2])
        except Exception:
            pass
        return kc[layer_idx], vc[layer_idx]

    def get_seq_length(self, layer_idx=0):
        kc = object.__getattribute__(self, "key_cache")
        return kc[0].shape[-2] if (kc and kc[0] is not None) else 0

    def get_max_cache_shape(self):
        return None

    def to_legacy_cache(self):
        kc = object.__getattribute__(self, "key_cache")
        vc = object.__getattribute__(self, "value_cache")
        return tuple((k, v) for k, v in zip(kc, vc) if k is not None)

    def __len__(self):
        return len(object.__getattribute__(self, "key_cache"))


def _extract_kv_pairs(c):
    if isinstance(c, _OurCache):
        kc = object.__getattribute__(c, "key_cache")
        vc = object.__getattribute__(c, "value_cache")
        return [(k, v) for k, v in zip(kc, vc) if k is not None]
    if hasattr(c, "key_cache") and isinstance(c.key_cache, list):
        return [(k, v) for k, v in zip(c.key_cache, c.value_cache) if k is not None]
    if hasattr(c, "to_legacy_cache"):
        try:
            leg = c.to_legacy_cache()
            if leg: return [(t[0], t[1]) for t in leg]
        except Exception:
            pass
    return []

def ensure_dynamic_cache(past_kv):
    if past_kv is None:         return _OurCache()
    if isinstance(past_kv, _OurCache): return past_kv
    cache = _OurCache()
    for i, (k, v) in enumerate(_extract_kv_pairs(past_kv)):
        cache.update(k, v, i)
    return cache

def cache_seq_len(c) -> int:
    kc = (object.__getattribute__(c, "key_cache") if isinstance(c, _OurCache)
          else getattr(c, "key_cache", None))
    if isinstance(kc, list) and kc and kc[0] is not None:
        return kc[0].shape[-2]
    try: return c.get_seq_length()
    except Exception: return 0

def num_cache_layers(c) -> int:
    kc = (object.__getattribute__(c, "key_cache") if isinstance(c, _OurCache)
          else getattr(c, "key_cache", None))
    return len(kc) if isinstance(kc, list) else len(_extract_kv_pairs(c))

def get_cache_tensors(c, layer_idx: int):
    if isinstance(c, _OurCache):
        kc = object.__getattribute__(c, "key_cache")
        vc = object.__getattribute__(c, "value_cache")
        return kc[layer_idx], vc[layer_idx]
    return c.key_cache[layer_idx], c.value_cache[layer_idx]

def slice_dynamic_cache(cache_obj, keep_idx_per_layer: List[torch.Tensor]) -> "_OurCache":
    new_cache = _OurCache()
    for li in range(num_cache_layers(cache_obj)):
        k, v  = get_cache_tensors(cache_obj, li)
        idx   = keep_idx_per_layer[li].to(k.device)
        new_cache.update(k.index_select(-2, idx), v.index_select(-2, idx), li)
    return new_cache

print("Cache utilities loaded.")

In [ ]:
# ─── LongBench data ───────────────────────────────────────────────────────────
def load_longbench_task(task_name: str, max_examples: int = MAX_EXAMPLES_PER_TASK):
    try:
        ds = load_dataset("THUDM/LongBench", task_name, split="test",
                          trust_remote_code=True)
        ds = ds.select(range(min(max_examples, len(ds))))
        print(f"  {task_name}: {len(ds)} (HF)")
        return ds
    except Exception as e:
        print(f"  HF failed ({e}), trying local …")
    extract_dir = DATA_DIR / "raw"
    jl = list(extract_dir.rglob(f"{task_name}.jsonl"))
    if not jl:
        zp = hf_hub_download(repo_id="THUDM/LongBench", repo_type="dataset",
                             filename="data.zip")
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zp, "r") as z:
            z.extractall(extract_dir)
        jl = list(extract_dir.rglob(f"{task_name}.jsonl"))
    ds = load_dataset("json", data_files=str(jl[0]), split="train")
    ds = ds.select(range(min(max_examples, len(ds))))
    print(f"  {task_name}: {len(ds)} (local)")
    return ds

def get_field(ex, names, default=""):
    for n in names:
        if n in ex and ex[n] is not None: return ex[n]
    return default

def build_prompt(ex: dict, task_name: str,
                 max_tok: int = MAX_INPUT_TOKENS) -> str:
    ctx = get_field(ex, ["context","article","passage"], "")
    q   = get_field(ex, ["input","question","query","instruction"], "")
    if task_name == "gov_report":
        uc = f"Read the following document and write a concise summary.\n\nDocument:\n{ctx}\n\nSummary:"
    else:
        uc = (f"Read the context below and answer the question concisely. "
              f"Give the best supported answer only.\n\nContext:\n{ctx}"
              f"\n\nQuestion:\n{q}\n\nAnswer:")
    msgs = [{"role": "user", "content": uc}]
    txt  = tokenizer.apply_chat_template(msgs, tokenize=False,
                                          add_generation_prompt=True)
    ids  = tokenizer(txt, return_tensors="pt",
                     add_special_tokens=False)["input_ids"][0]
    if len(ids) <= max_tok:
        return txt
    # head + tail truncation of context only
    if task_name == "gov_report":
        pfx, sfx = "Read the following document and write a concise summary.\n\nDocument:\n", "\n\nSummary:"
    else:
        pfx = ("Read the context below and answer the question concisely. "
               "Give the best supported answer only.\n\nContext:\n")
        sfx = f"\n\nQuestion:\n{q}\n\nAnswer:"
    avail    = max(256, max_tok - len(tokenizer(pfx, add_special_tokens=False)["input_ids"])
                             - len(tokenizer(sfx, add_special_tokens=False)["input_ids"]) - 50)
    ctx_ids  = tokenizer(ctx, add_special_tokens=False)["input_ids"]
    if len(ctx_ids) > avail:
        h = ctx_ids[:avail//2]; t = ctx_ids[-(avail-len(h)):]
        ctx_ids = h + t
    trunc = tokenizer.decode(ctx_ids, skip_special_tokens=True)
    if task_name == "gov_report":
        uc = f"Read the following document and write a concise summary.\n\nDocument:\n{trunc}\n\nSummary:"
    else:
        uc = (f"Read the context below and answer the question concisely. "
              f"Give the best supported answer only.\n\nContext:\n{trunc}"
              f"\n\nQuestion:\n{q}\n\nAnswer:")
    msgs = [{"role": "user", "content": uc}]
    return tokenizer.apply_chat_template(msgs, tokenize=False,
                                          add_generation_prompt=True)

# ─── Metrics ──────────────────────────────────────────────────────────────────
_ROUGE = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def normalize_answer(s: str) -> str:
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    return " ".join(s.split())

def qa_f1(pred: str, gold: str) -> float:
    p = normalize_answer(pred).split(); g = normalize_answer(gold).split()
    if not p or not g: return 0.0
    common = Counter(p) & Counter(g); n = sum(common.values())
    if not n: return 0.0
    return 2*(n/len(p))*(n/len(g)) / ((n/len(p))+(n/len(g)))

def rouge_l(pred: str, gold: str) -> float:
    try: return _ROUGE.score(gold, pred)["rougeL"].fmeasure
    except Exception: return 0.0

def score_prediction(task_name: str, pred: str, answers) -> Tuple[float, str]:
    if isinstance(answers, str): answers = [answers]
    answers = [str(a) for a in answers] if answers else [""]
    if task_name == "gov_report":
        return max(rouge_l(pred, a) for a in answers), "rougeL"
    return max(qa_f1(pred, a)   for a in answers), "f1"

print("Data + prompt + metrics loaded.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# COMPRESSOR CLASSES  (all 8 compression methods share the same interface)
#
#   update_importance(attentions, cache_obj)   — called after every forward pass
#   compress(cache_obj) -> cache_obj           — evict low-priority entries
#
# The generation function anchors _kv_budget = int(prefill_len * budget_ratio)
# so all methods maintain a stable target size throughout decoding.
# ═══════════════════════════════════════════════════════════════════════════════

# ─── 1. StreamingLLM ──────────────────────────────────────────────────────────

class StreamingLLMCompressor:
    """
    Efficient Streaming Language Models with Attention Sinks (Xiao et al., 2023)
    https://arxiv.org/abs/2309.17453

    Keeps the first `sink_size` tokens (attention sinks) and a FIFO sliding
    window of the most-recent tokens.  No attention-weight tracking needed.
    """
    def __init__(self, budget_ratio: float = 0.5, sink_size: int = 4):
        self.budget_ratio = budget_ratio
        self.sink_size    = sink_size

    def update_importance(self, attentions, cache_obj): pass

    def compress(self, cache_obj) -> "_OurCache":
        seq_len = cache_seq_len(cache_obj)
        budget  = getattr(self, "_kv_budget",
                          max(self.sink_size + 1, int(seq_len * self.budget_ratio)))
        if seq_len <= budget:
            return cache_obj
        window_size  = max(1, budget - self.sink_size)
        sink_actual  = budget - window_size          # = min(sink_size, budget)
        dev = get_cache_tensors(cache_obj, 0)[0].device
        sink_idx   = torch.arange(0, sink_actual, device=dev)
        win_start  = max(sink_actual, seq_len - window_size)
        win_idx    = torch.arange(win_start, seq_len, device=dev)
        keep_idx   = torch.cat([sink_idx, win_idx])
        n = num_cache_layers(cache_obj)
        return slice_dynamic_cache(cache_obj, [keep_idx] * n)


# ─── 2. H2O  ──────────────────────────────────────────────────────────────────

class H2OCompressor:
    """
    H2O: Heavy-Hitter Oracle (Zhang et al., NeurIPS 2023)
    https://arxiv.org/abs/2306.14048

    Accumulates per-position attention importance across all forward passes.
    Budget is split equally between heavy-hitter top-K and a recency window.
    Compresses at every decode step (decode-time eviction).
    """
    def __init__(self, budget_ratio: float = 0.5,
                 hh_fraction: float = 0.5,
                 recent_window: int = RECENT_WINDOW):
        self.budget_ratio  = budget_ratio
        self.hh_fraction   = hh_fraction   # fraction of budget for heavy hitters
        self.recent_window = recent_window
        self.importance: Optional[List[torch.Tensor]] = None

    def update_importance(self, attentions, cache_obj):
        n = num_cache_layers(cache_obj)
        if self.importance is None:
            k0, _ = get_cache_tensors(cache_obj, 0)
            self.importance = [
                torch.zeros(cache_seq_len(cache_obj), device=k0.device, dtype=torch.float32)
                for _ in range(n)]
        for li, attn in enumerate(attentions):
            score = attn.mean(dim=1)[0, -1, :].float()
            cur = self.importance[li]
            if score.shape[0] > cur.shape[0]:
                pad = torch.zeros(score.shape[0] - cur.shape[0], device=cur.device)
                cur = torch.cat([cur, pad])
            cur[:score.shape[0]] += score
            self.importance[li] = cur

    def compress(self, cache_obj) -> "_OurCache":
        seq_len    = cache_seq_len(cache_obj)
        budget     = getattr(self, "_kv_budget",
                             max(self.recent_window, int(seq_len * self.budget_ratio)))
        if budget >= seq_len:
            return cache_obj
        recent_sz  = max(1, int(budget * (1 - self.hh_fraction)))
        hh_sz      = budget - recent_sz
        keep_idx_per_layer = []
        for li in range(num_cache_layers(cache_obj)):
            imp = self.importance[li][:seq_len]
            rec_start = max(0, seq_len - recent_sz)
            rec_idx   = torch.arange(rec_start, seq_len, device=imp.device)
            if hh_sz <= 0 or rec_start == 0:
                keep_idx = rec_idx
            else:
                k = min(hh_sz, rec_start)
                hh_idx = torch.topk(imp[:rec_start], k=k).indices.sort().values
                keep_idx = torch.cat([hh_idx, rec_idx])
            keep_idx_per_layer.append(keep_idx)
        new_cache = slice_dynamic_cache(cache_obj, keep_idx_per_layer)
        self.importance = [
            self.importance[i].index_select(
                0, keep_idx_per_layer[i].to(self.importance[i].device))
            for i in range(len(self.importance))]
        return new_cache


# ─── 3. SnapKV  ───────────────────────────────────────────────────────────────

class SnapKVCompressor:
    """
    SnapKV: LLM Knows What You are Looking for before Generation (Li et al., 2024)
    https://arxiv.org/abs/2404.14469

    Compresses the KV cache **once** after prefill using the attention patterns
    from the last `observation_window` query positions (max-pooled).
    No eviction during decode — cache grows freely after initial compression.
    """
    def __init__(self, budget_ratio: float = 0.5,
                 recent_window: int = RECENT_WINDOW,
                 observation_window: int = 32):
        self.budget_ratio       = budget_ratio
        self.recent_window      = recent_window
        self.observation_window = observation_window
        self.importance: Optional[List[torch.Tensor]] = None
        self._compressed        = False

    def update_importance(self, attentions, cache_obj):
        if self._compressed:
            return           # only use prefill attention
        self.importance = []
        for attn in attentions:
            # attn: (batch, heads, q_len, kv_len) — q_len == kv_len during prefill
            obs   = min(self.observation_window, attn.shape[2])
            score = attn[:, :, -obs:, :].max(dim=2).values  # (B, H, kv_len)
            self.importance.append(score.mean(dim=1)[0].float())  # (kv_len,)

    def compress(self, cache_obj) -> "_OurCache":
        if self._compressed:
            return cache_obj
        seq_len    = cache_seq_len(cache_obj)
        keep_total = getattr(self, "_kv_budget",
                             max(self.recent_window, int(seq_len * self.budget_ratio)))
        if keep_total >= seq_len or self.importance is None:
            self._compressed = True
            return cache_obj
        keep_idx_per_layer = []
        for li in range(num_cache_layers(cache_obj)):
            imp       = self.importance[li][:seq_len]
            rec_start = max(0, seq_len - self.recent_window)
            rec_idx   = torch.arange(rec_start, seq_len, device=imp.device)
            old_bud   = keep_total - len(rec_idx)
            if old_bud <= 0:
                keep_idx = rec_idx
            else:
                k = min(old_bud, rec_start)
                old_idx  = torch.topk(imp[:rec_start], k=k).indices.sort().values
                keep_idx = torch.cat([old_idx, rec_idx])
            keep_idx_per_layer.append(keep_idx)
        self._compressed = True
        return slice_dynamic_cache(cache_obj, keep_idx_per_layer)


# ─── 4. KNorm  ────────────────────────────────────────────────────────────────

class KNormCompressor:
    """
    KNorm: key-norm importance (Devoto et al., EMNLP 2024)
    https://aclanthology.org/2024.emnlp-main.1027/

    Uses the L2 norm of each key vector (averaged across heads) as the
    importance signal.  Compresses once after prefill, no decode eviction.
    """
    def __init__(self, budget_ratio: float = 0.5,
                 recent_window: int = RECENT_WINDOW):
        self.budget_ratio  = budget_ratio
        self.recent_window = recent_window
        self._compressed   = False

    def update_importance(self, attentions, cache_obj): pass  # uses key norms

    def compress(self, cache_obj) -> "_OurCache":
        if self._compressed:
            return cache_obj
        seq_len    = cache_seq_len(cache_obj)
        keep_total = getattr(self, "_kv_budget",
                             max(self.recent_window, int(seq_len * self.budget_ratio)))
        if keep_total >= seq_len:
            self._compressed = True
            return cache_obj
        keep_idx_per_layer = []
        for li in range(num_cache_layers(cache_obj)):
            keys, _ = get_cache_tensors(cache_obj, li)
            # keys: (batch, n_kv_heads, seq_len, head_dim)
            key_norms = keys[0].norm(dim=-1).mean(dim=0)[:seq_len].float()
            rec_start = max(0, seq_len - self.recent_window)
            rec_idx   = torch.arange(rec_start, seq_len, device=keys.device)
            old_bud   = keep_total - len(rec_idx)
            if old_bud <= 0:
                keep_idx = rec_idx
            else:
                k = min(old_bud, rec_start)
                old_idx  = torch.topk(key_norms[:rec_start], k=k).indices.sort().values
                keep_idx = torch.cat([old_idx, rec_idx])
            keep_idx_per_layer.append(keep_idx)
        self._compressed = True
        return slice_dynamic_cache(cache_obj, keep_idx_per_layer)


# ─── 5. ScissorHands  ─────────────────────────────────────────────────────────

class ScissorHandsCompressor:
    """
    ScissorHands: Scraping Off the Unwanted Memories (Liu et al., 2024)

    Accumulates per-position cumulative attention importance (same mechanism
    as H2O) with a uniform budget split: top-K old tokens + recency window.
    Budget is anchored to prefill length for stable per-step eviction.
    """
    def __init__(self, budget_ratio: float = 0.5,
                 recent_window: int = RECENT_WINDOW):
        self.budget_ratio  = budget_ratio
        self.recent_window = recent_window
        self.importance: Optional[List[torch.Tensor]] = None

    def update_importance(self, attentions, cache_obj):
        n = num_cache_layers(cache_obj)
        if self.importance is None:
            k0, _ = get_cache_tensors(cache_obj, 0)
            self.importance = [
                torch.zeros(cache_seq_len(cache_obj), device=k0.device, dtype=torch.float32)
                for _ in range(n)]
        for li, attn in enumerate(attentions):
            score = attn.mean(dim=1)[0, -1, :].float()
            cur = self.importance[li]
            if score.shape[0] > cur.shape[0]:
                cur = torch.cat([cur, torch.zeros(score.shape[0]-cur.shape[0],
                                                  device=cur.device)])
            cur[:score.shape[0]] += score
            self.importance[li] = cur

    def compress(self, cache_obj) -> "_OurCache":
        seq_len    = cache_seq_len(cache_obj)
        budget     = getattr(self, "_kv_budget",
                             max(self.recent_window, int(seq_len * self.budget_ratio)))
        if budget >= seq_len:
            return cache_obj
        keep_idx_per_layer = []
        for li in range(num_cache_layers(cache_obj)):
            imp       = self.importance[li][:seq_len]
            rec_start = max(0, seq_len - self.recent_window)
            rec_idx   = torch.arange(rec_start, seq_len, device=imp.device)
            old_bud   = budget - len(rec_idx)
            if old_bud <= 0 or rec_start == 0:
                keep_idx = rec_idx
            else:
                k        = min(old_bud, rec_start)
                old_idx  = torch.topk(imp[:rec_start], k=k).indices.sort().values
                keep_idx = torch.cat([old_idx, rec_idx])
            keep_idx_per_layer.append(keep_idx)
        new_cache = slice_dynamic_cache(cache_obj, keep_idx_per_layer)
        self.importance = [
            self.importance[i].index_select(
                0, keep_idx_per_layer[i].to(self.importance[i].device))
            for i in range(len(self.importance))]
        return new_cache


# ─── 6. AdaKV  ────────────────────────────────────────────────────────────────

class AdaKVCompressor:
    """
    AdaKV: Head-Adaptive KV Cache Compression (Fu et al., 2024)

    Allocates a per-layer budget proportional to each layer's attention
    concentration (entropy-based).  Layers with peaky, focused attention
    get a tighter budget; layers with diffuse attention get more slack.
    """
    def __init__(self, budget_ratio: float = 0.5,
                 recent_window: int = RECENT_WINDOW):
        self.budget_ratio  = budget_ratio
        self.recent_window = recent_window
        self.importance: Optional[List[torch.Tensor]] = None
        self.layer_strength: Optional[List[float]]    = None

    @staticmethod
    def _concentration(score: torch.Tensor) -> float:
        score = score.clamp_min(1e-8)
        p = score / score.sum()
        entropy = -(p * p.log()).sum()
        max_ent = math.log(len(p) + 1e-8)
        return max(1e-4, 1.0 - float(entropy / max_ent) if max_ent > 0 else 1.0)

    def update_importance(self, attentions, cache_obj):
        n = num_cache_layers(cache_obj)
        if self.importance is None:
            k0, _ = get_cache_tensors(cache_obj, 0)
            self.importance    = [
                torch.zeros(cache_seq_len(cache_obj), device=k0.device, dtype=torch.float32)
                for _ in range(n)]
            self.layer_strength = [1.0] * n
        for li, attn in enumerate(attentions):
            score = attn.mean(dim=1)[0, -1, :].float()
            cur = self.importance[li]
            if score.shape[0] > cur.shape[0]:
                cur = torch.cat([cur, torch.zeros(score.shape[0]-cur.shape[0],
                                                  device=cur.device)])
            cur[:score.shape[0]] += score
            self.importance[li] = cur
            conc = self._concentration(score.detach())
            self.layer_strength[li] = 0.9*self.layer_strength[li] + 0.1*conc

    def compress(self, cache_obj) -> "_OurCache":
        seq_len    = cache_seq_len(cache_obj)
        n          = num_cache_layers(cache_obj)
        total_keep = getattr(self, "_kv_budget",
                             max(self.recent_window, int(seq_len * self.budget_ratio)))
        if total_keep >= seq_len:
            return cache_obj
        strength = np.array(self.layer_strength, dtype=np.float64)
        strength = strength / strength.sum()
        layer_budgets = [
            max(self.recent_window, min(seq_len,
                int(round(total_keep * (0.5 + 0.5 * n * s)))))
            for s in strength]
        keep_idx_per_layer = []
        for li in range(n):
            imp       = self.importance[li][:seq_len]
            bud       = layer_budgets[li]
            rec_start = max(0, seq_len - self.recent_window)
            rec_idx   = torch.arange(rec_start, seq_len, device=imp.device)
            old_bud   = bud - len(rec_idx)
            if old_bud <= 0 or rec_start == 0:
                keep_idx = rec_idx
            else:
                k        = min(old_bud, rec_start)
                old_idx  = torch.topk(imp[:rec_start], k=k).indices.sort().values
                keep_idx = torch.cat([old_idx, rec_idx])
            keep_idx_per_layer.append(keep_idx)
        new_cache = slice_dynamic_cache(cache_obj, keep_idx_per_layer)
        self.importance = [
            self.importance[i].index_select(
                0, keep_idx_per_layer[i].to(self.importance[i].device))
            for i in range(n)]
        return new_cache


# ─── 7. RKV  ──────────────────────────────────────────────────────────────────

class RKVCompressor:
    """
    R-KV: Redundancy-aware KV Cache Compression (NeurIPS 2025)

    Joint scoring: importance × (1 – max cosine similarity with recent-window
    neighbours).  Tokens that are both attended-to AND unique in key-space are
    retained; repeated / redundant tokens are evicted first.
    """
    def __init__(self, budget_ratio: float = 0.5,
                 recent_window: int = RECENT_WINDOW):
        self.budget_ratio  = budget_ratio
        self.recent_window = recent_window
        self.importance: Optional[List[torch.Tensor]] = None

    def update_importance(self, attentions, cache_obj):
        if self.importance is None:
            k0, _ = get_cache_tensors(cache_obj, 0)
            self.importance = [
                torch.zeros(cache_seq_len(cache_obj), device=k0.device, dtype=torch.float32)
                for _ in range(num_cache_layers(cache_obj))]
        for li, attn in enumerate(attentions):
            score = attn.mean(dim=1)[0, -1, :].float()
            cur = self.importance[li]
            if score.shape[0] > cur.shape[0]:
                cur = torch.cat([cur, torch.zeros(score.shape[0]-cur.shape[0],
                                                  device=cur.device)])
            cur[:score.shape[0]] += score
            self.importance[li] = cur

    def _non_redundancy(self, keys: torch.Tensor, old_end: int) -> torch.Tensor:
        if old_end <= 1:
            return torch.ones(old_end, device=keys.device)
        k      = keys[0].mean(dim=0)[:old_end].float()
        k_norm = F.normalize(k, dim=-1)
        sim    = k_norm @ k_norm.T
        W      = self.recent_window
        idx    = torch.arange(old_end, device=k.device)
        valid  = (idx.unsqueeze(0) < idx.unsqueeze(1)) & \
                 (idx.unsqueeze(0) >= idx.unsqueeze(1) - W)
        sim.masked_fill_(~valid, -1.0)
        max_sim = sim.max(dim=1).values.clamp(0.0, 1.0)
        max_sim[0] = 0.0
        return 1.0 - max_sim

    def compress(self, cache_obj) -> "_OurCache":
        seq_len    = cache_seq_len(cache_obj)
        budget     = getattr(self, "_kv_budget",
                             max(self.recent_window, int(seq_len * self.budget_ratio)))
        if budget >= seq_len:
            return cache_obj
        rec_start  = max(0, seq_len - self.recent_window)
        rec_idx    = torch.arange(rec_start, seq_len)
        old_bud    = budget - len(rec_idx)
        keep_idx_per_layer = []
        for li in range(num_cache_layers(cache_obj)):
            if old_bud <= 0 or rec_start == 0:
                keep_idx = rec_idx
            else:
                imp      = self.importance[li][:seq_len]
                keys, _  = get_cache_tensors(cache_obj, li)
                nr       = self._non_redundancy(keys, rec_start)
                joint    = imp[:rec_start] * (0.5 + 0.5 * nr)
                k        = min(old_bud, rec_start)
                old_idx  = torch.topk(joint, k=k).indices.sort().values.cpu()
                keep_idx = torch.cat([old_idx, rec_idx.cpu()])
            keep_idx_per_layer.append(keep_idx)
        new_cache = slice_dynamic_cache(cache_obj, keep_idx_per_layer)
        self.importance = [
            self.importance[i].index_select(
                0, keep_idx_per_layer[i].to(self.importance[i].device))
            for i in range(len(self.importance))]
        return new_cache


# ─── 8. ChunkKV  ──────────────────────────────────────────────────────────────

class ChunkKVCompressor:
    """
    ChunkKV: Semantic-Preserving KV Cache Compression (NeurIPS 2025)

    Scores tokens in fixed-size semantic chunks (mean importance per chunk).
    The same chunk selection is applied to ALL layers (layer-index reuse),
    reducing per-layer overhead and preserving semantic coherence.
    """
    def __init__(self, budget_ratio: float = 0.5,
                 recent_window: int = RECENT_WINDOW,
                 chunk_size: int = 64):
        self.budget_ratio  = budget_ratio
        self.recent_window = recent_window
        self.chunk_size    = chunk_size
        self.importance: Optional[List[torch.Tensor]] = None

    def update_importance(self, attentions, cache_obj):
        if self.importance is None:
            k0, _ = get_cache_tensors(cache_obj, 0)
            self.importance = [
                torch.zeros(cache_seq_len(cache_obj), device=k0.device, dtype=torch.float32)
                for _ in range(num_cache_layers(cache_obj))]
        for li, attn in enumerate(attentions):
            score = attn.mean(dim=1)[0, -1, :].float()
            cur = self.importance[li]
            if score.shape[0] > cur.shape[0]:
                cur = torch.cat([cur, torch.zeros(score.shape[0]-cur.shape[0],
                                                  device=cur.device)])
            cur[:score.shape[0]] += score
            self.importance[li] = cur

    def compress(self, cache_obj) -> "_OurCache":
        seq_len    = cache_seq_len(cache_obj)
        n          = num_cache_layers(cache_obj)
        budget     = getattr(self, "_kv_budget",
                             max(self.recent_window, int(seq_len * self.budget_ratio)))
        if budget >= seq_len:
            return cache_obj
        rec_start = max(0, seq_len - self.recent_window)
        old_bud   = max(0, budget - self.recent_window)
        # Average importance across layers → shared chunk selection for all layers
        avg_imp   = torch.stack([self.importance[i][:seq_len] for i in range(n)]).mean(0)
        if rec_start > 0 and old_bud > 0:
            n_chunks = max(1, (rec_start + self.chunk_size - 1) // self.chunk_size)
            chunk_sc = []
            for c in range(n_chunks):
                s, e = c*self.chunk_size, min((c+1)*self.chunk_size, rec_start)
                chunk_sc.append((avg_imp[s:e].mean().item(), s, e))
            chunk_sc.sort(key=lambda x: x[0], reverse=True)
            k_chunks = max(1, round(n_chunks * (old_bud / max(rec_start, 1))))
            old_pos  = sorted(set(
                pos for _, s, e in chunk_sc[:k_chunks] for pos in range(s, e)))
        else:
            old_pos = []
        all_idx = sorted(set(old_pos + list(range(rec_start, seq_len))))
        if len(all_idx) > budget:
            all_idx = all_idx[-budget:]
        keep_idx_global = torch.tensor(all_idx, dtype=torch.long)
        keep_idx_per_layer = [keep_idx_global.clone() for _ in range(n)]
        new_cache = slice_dynamic_cache(cache_obj, keep_idx_per_layer)
        self.importance = [
            self.importance[i].index_select(
                0, keep_idx_per_layer[i].to(self.importance[i].device))
            for i in range(n)]
        return new_cache


print("All 8 compressor classes loaded.")

In [ ]:
# ─── Generation functions ─────────────────────────────────────────────────────

@torch.no_grad()
def generate_baseline(prompt: str, max_new_tokens: int = MAX_NEW_TOKENS) -> dict:
    """Full-cache greedy generation (no eviction)."""
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=MAX_INPUT_TOKENS).to(model.device)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()

    out  = model(**inputs, use_cache=True, past_key_values=_OurCache(),
                 output_attentions=False, return_dict=True)
    past = ensure_dynamic_cache(out.past_key_values)
    nxt  = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    gen  = [nxt.item()]; lens = [cache_seq_len(past)]

    for _ in range(max_new_tokens - 1):
        out  = model(input_ids=nxt, past_key_values=past, use_cache=True,
                     output_attentions=False, return_dict=True)
        past = ensure_dynamic_cache(out.past_key_values)
        nxt  = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        tok  = nxt.item(); gen.append(tok); lens.append(cache_seq_len(past))
        if tok == tokenizer.eos_token_id: break

    if torch.cuda.is_available(): torch.cuda.synchronize()
    t1 = time.perf_counter()
    lat = t1 - t0
    rec = {"text": tokenizer.decode(gen, skip_special_tokens=True).strip(),
           "latency": lat, "tokens_per_sec": len(gen)/max(lat,1e-6),
           "generated_tokens": len(gen),
           "final_cache_len": cache_seq_len(past),
           "avg_cache_len": float(np.mean(lens))}
    del out, past, inputs, lens, gen
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return rec


@torch.no_grad()
def generate_with_compressor(prompt: str, compressor,
                              max_new_tokens: int = MAX_NEW_TOKENS) -> dict:
    """
    Step-by-step greedy generation with KV-cache eviction.

    The compressor's _kv_budget is anchored to the prefill length so every
    decode step evicts back to the same absolute target size.
    SnapKV / KNorm mark themselves done after the first compression and
    subsequently let the cache grow freely (no decode-time eviction).
    """
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=MAX_INPUT_TOKENS).to(model.device)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()

    # Prefill — pass _OurCache() directly so the model writes into our lists
    out  = model(**inputs, use_cache=True, past_key_values=_OurCache(),
                 output_attentions=True, return_dict=True)
    past = ensure_dynamic_cache(out.past_key_values)

    # Anchor budget to prefill length (applies to all compressors uniformly)
    prefill_len = cache_seq_len(past)
    rw = getattr(compressor, "recent_window", getattr(compressor, "sink_size", 4))
    compressor._kv_budget = max(rw, int(prefill_len * compressor.budget_ratio))

    compressor.update_importance(out.attentions, past)
    past = compressor.compress(past)
    nxt  = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    del out
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    gen  = [nxt.item()]; lens = [cache_seq_len(past)]

    for _ in range(max_new_tokens - 1):
        out  = model(input_ids=nxt, past_key_values=past, use_cache=True,
                     output_attentions=True, return_dict=True)
        past = ensure_dynamic_cache(out.past_key_values)
        compressor.update_importance(out.attentions, past)
        past = compressor.compress(past)
        nxt  = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        tok  = nxt.item(); gen.append(tok); lens.append(cache_seq_len(past))
        if tok == tokenizer.eos_token_id: break

    if torch.cuda.is_available(): torch.cuda.synchronize()
    t1  = time.perf_counter()
    lat = t1 - t0
    rec = {"text": tokenizer.decode(gen, skip_special_tokens=True).strip(),
           "latency": lat, "tokens_per_sec": len(gen)/max(lat,1e-6),
           "generated_tokens": len(gen),
           "final_cache_len": cache_seq_len(past),
           "avg_cache_len": float(np.mean(lens))}
    del out, past, inputs, lens, gen
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return rec


print("Generation functions loaded.")

In [ ]:
# ─── Compressor factory ───────────────────────────────────────────────────────

def build_compressor(method: str, budget: float):
    """Return a fresh compressor instance for (method, budget)."""
    if method == "baseline":     return None
    if method == "streamingllm": return StreamingLLMCompressor(budget_ratio=budget)
    if method == "h2o":          return H2OCompressor(budget_ratio=budget,
                                                      recent_window=RECENT_WINDOW)
    if method == "snapkv":       return SnapKVCompressor(budget_ratio=budget,
                                                         recent_window=RECENT_WINDOW)
    if method == "knorm":        return KNormCompressor(budget_ratio=budget,
                                                        recent_window=RECENT_WINDOW)
    if method == "scissorhands": return ScissorHandsCompressor(budget_ratio=budget,
                                                               recent_window=RECENT_WINDOW)
    if method == "adakv":        return AdaKVCompressor(budget_ratio=budget,
                                                        recent_window=RECENT_WINDOW)
    if method == "rkv":          return RKVCompressor(budget_ratio=budget,
                                                      recent_window=RECENT_WINDOW)
    if method == "chunkkv":      return ChunkKVCompressor(budget_ratio=budget,
                                                          recent_window=RECENT_WINDOW)
    raise ValueError(f"Unknown method: {method}")


def run_one_example(task_name: str, example: dict,
                    method: str, budget: float) -> dict:
    prompt  = build_prompt(example, task_name)
    answers = get_field(example, ["answers","answer"], [])
    if isinstance(answers, str): answers = [answers]

    if method == "baseline":
        result = generate_baseline(prompt)
    else:
        comp   = build_compressor(method, budget)
        result = generate_with_compressor(prompt, comp)

    score, metric = score_prediction(task_name, result["text"], answers)
    return {
        "task":             task_name,
        "method":           method,
        "kv_budget":        budget,
        "metric":           metric,
        "score":            score,
        "latency":          result["latency"],
        "tokens_per_sec":   result["tokens_per_sec"],
        "generated_tokens": result["generated_tokens"],
        "final_cache_len":  result["final_cache_len"],
        "avg_cache_len":    result["avg_cache_len"],
        "prediction":       result["text"],
        "answers":          json.dumps(answers, ensure_ascii=False),
    }

# ── Quick smoke test (1 example, 4 methods) ───────────────────────────────────
print("Loading smoke-test example …")
smoke_ds  = load_longbench_task("hotpotqa", max_examples=1)
smoke_ex  = smoke_ds[0]
smoke_mth = [("baseline", 1.0), ("h2o", 0.2), ("snapkv", 0.2), ("rkv", 0.2)]

print("\n── Smoke test ───────────────────────────────────────────────────────")
for m, b in smoke_mth:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    r = run_one_example("hotpotqa", smoke_ex, m, b)
    print(f"  {m:12s} budget={b}  score={r['score']:.3f}  "
          f"lat={r['latency']:.2f}s  cache={r['final_cache_len']}  "
          f'pred="{r["prediction"][:60]}"')
print("Smoke test passed!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FULL EVALUATION
# Iterates task → method → budget → example.
# Results are appended to a CSV after each example for resume support.
# ═══════════════════════════════════════════════════════════════════════════════

RAW_CSV = OUTPUT_DIR / "all_results.csv"

METHODS_CONFIG = {
    "baseline":     [1.0],
    "streamingllm": BUDGETS,
    "h2o":          BUDGETS,
    "snapkv":       BUDGETS,
    "knorm":        BUDGETS,
    "scissorhands": BUDGETS,
    "adakv":        BUDGETS,
    "rkv":          BUDGETS,
    "chunkkv":      BUDGETS,
}

def load_completed_keys(csv_path: Path) -> set:
    if not csv_path.exists():
        return set()
    df = pd.read_csv(csv_path)
    return {(r.task, r.method, float(r.kv_budget), int(r.example_id))
            for _, r in df.iterrows()}

def append_row(row: dict, csv_path: Path):
    pd.DataFrame([row]).to_csv(csv_path, mode="a",
                               header=not csv_path.exists(), index=False)

completed = load_completed_keys(RAW_CSV)
print(f"Resuming from {len(completed)} completed examples.")

for task_name in TASKS:
    print(f"\n{'='*22} {task_name} {'='*22}")
    ds = load_longbench_task(task_name)
    if ds is None: continue

    for method, budgets in METHODS_CONFIG.items():
        for budget in budgets:
            n_done = sum(1 for k in completed
                         if k[0]==task_name and k[1]==method and k[2]==budget)
            todo   = len(ds) - n_done
            if todo <= 0:
                print(f"  {method:12s} budget={budget}  [already done]")
                continue
            print(f"  {method:12s} budget={budget}  ({todo} examples)")
            for i, ex in enumerate(tqdm(ds, desc=f"{method}@{budget}", leave=False)):
                key = (task_name, method, budget, i)
                if key in completed: continue
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                try:
                    row = run_one_example(task_name, ex, method, budget)
                    row["example_id"] = i
                    append_row(row, RAW_CSV)
                    completed.add(key)
                except RuntimeError as exc:
                    print(f"  !! SKIPPED {task_name} ex={i} {method}@{budget}: {exc}")
                    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"\nDone. Results saved to {RAW_CSV}  ({len(completed)} total rows)")

In [ ]:
# ─── Load & aggregate results ─────────────────────────────────────────────────
results_df = pd.read_csv(RAW_CSV)
print(f"Loaded {len(results_df)} rows, {results_df['method'].nunique()} methods, "
      f"{results_df['task'].nunique()} tasks")

summary_df = (
    results_df.groupby(["task","method","kv_budget","metric"], as_index=False)
    .agg(score_mean=("score","mean"), latency_mean=("latency","mean"),
         tps_mean=("tokens_per_sec","mean"),
         final_cache_mean=("final_cache_len","mean"))
    .sort_values(["task","method","kv_budget"])
)

overall_df = (
    results_df.groupby(["method","kv_budget"], as_index=False)
    .agg(score_mean=("score","mean"), latency_mean=("latency","mean"),
         tps_mean=("tokens_per_sec","mean"),
         final_cache_mean=("final_cache_len","mean"))
    .sort_values(["method","kv_budget"])
)

summary_df.to_csv(OUTPUT_DIR / "task_summary.csv", index=False)
overall_df.to_csv(OUTPUT_DIR / "overall_summary.csv", index=False)

# Pivot table: rows=task, columns=(method, budget)
pivot = summary_df.pivot_table(
    index="task", columns=["method","kv_budget"], values="score_mean")
pivot.to_csv(OUTPUT_DIR / "score_pivot.csv")

print("\n── Overall averages ─────────────────────────────────────────────────────")
print(overall_df.to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PLOTS
# ═══════════════════════════════════════════════════════════════════════════════

METHOD_STYLES = {
    "baseline":     dict(color="#333333", marker="o", ls="-",   lw=2.5),
    "streamingllm": dict(color="#1f77b4", marker="s", ls="--",  lw=1.8),
    "h2o":          dict(color="#ff7f0e", marker="^", ls="--",  lw=1.8),
    "snapkv":       dict(color="#2ca02c", marker="D", ls="-.",  lw=1.8),
    "knorm":        dict(color="#9467bd", marker="v", ls="-.",  lw=1.8),
    "scissorhands": dict(color="#8c564b", marker="P", ls=":",   lw=1.8),
    "adakv":        dict(color="#e377c2", marker="X", ls=":",   lw=1.8),
    "rkv":          dict(color="#17becf", marker="*", ls="-",   lw=1.8),
    "chunkkv":      dict(color="#bcbd22", marker="h", ls="-",   lw=1.8),
}

def _sub(method):
    return overall_df[overall_df["method"] == method].sort_values("kv_budget")


# ── Plot 1: Score vs KV budget ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for m in METHODS:
    sub = _sub(m)
    if sub.empty: continue
    ax.plot(sub["kv_budget"], sub["score_mean"],
            label=m, **METHOD_STYLES[m])
ax.set_xlabel("KV keep-ratio (fraction of prefill length retained)")
ax.set_ylabel("Macro-avg score across 6 LongBench tasks")
ax.set_title("Accuracy vs KV Cache Budget — 9 Methods")
ax.set_xticks(BUDGETS + [1.0])
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot_score_vs_budget.png", dpi=150)
plt.show()

# ── Plot 2: Latency vs KV budget ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for m in METHODS:
    sub = _sub(m)
    if sub.empty: continue
    ax.plot(sub["kv_budget"], sub["latency_mean"],
            label=m, **METHOD_STYLES[m])
ax.set_xlabel("KV keep-ratio")
ax.set_ylabel("Avg latency per sample (s)")
ax.set_title("Latency vs KV Cache Budget")
ax.set_xticks(BUDGETS + [1.0])
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot_latency_vs_budget.png", dpi=150)
plt.show()

# ── Plot 3: Throughput (tokens/sec) vs KV budget ──────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for m in METHODS:
    sub = _sub(m)
    if sub.empty: continue
    ax.plot(sub["kv_budget"], sub["tps_mean"],
            label=m, **METHOD_STYLES[m])
ax.set_xlabel("KV keep-ratio")
ax.set_ylabel("Avg throughput (tokens / second)")
ax.set_title("Throughput vs KV Cache Budget")
ax.set_xticks(BUDGETS + [1.0])
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot_throughput_vs_budget.png", dpi=150)
plt.show()

# ── Plot 4: Score–latency tradeoff scatter ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
for m in METHODS:
    sub = _sub(m)
    if sub.empty: continue
    st = METHOD_STYLES[m]
    ax.scatter(sub["latency_mean"], sub["score_mean"],
               color=st["color"], marker=st["marker"],
               s=90, zorder=3, label=m)
    # annotate each point with its budget
    for _, row in sub.iterrows():
        ax.annotate(f"{row['kv_budget']:.0%}",
                    (row["latency_mean"], row["score_mean"]),
                    textcoords="offset points", xytext=(5, 3),
                    fontsize=7, color=st["color"])
ax.set_xlabel("Avg latency per sample (s)")
ax.set_ylabel("Macro-avg score")
ax.set_title("Score–Latency Tradeoff (each point = one method × budget)")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot_score_latency_tradeoff.png", dpi=150)
plt.show()

# ── Plot 5: Per-task score heatmap at 20% budget ──────────────────────────────
budget_sel = 0.2
hm_data = summary_df[
    ((summary_df["kv_budget"] == budget_sel) |
     (summary_df["method"] == "baseline"))
].copy()
# For baseline (budget=1.0) assign kv_budget label "baseline"
hm_pivot = hm_data.pivot_table(index="task", columns="method",
                                values="score_mean", aggfunc="mean")
# Reorder columns
col_order = [m for m in METHODS if m in hm_pivot.columns]
hm_pivot  = hm_pivot[col_order]

fig, ax = plt.subplots(figsize=(13, 4))
im = ax.imshow(hm_pivot.values, aspect="auto", cmap="RdYlGn",
               vmin=0, vmax=hm_pivot.values.max())
ax.set_xticks(range(len(col_order))); ax.set_xticklabels(col_order, rotation=30, ha="right")
ax.set_yticks(range(len(hm_pivot.index))); ax.set_yticklabels(hm_pivot.index)
for r in range(hm_pivot.shape[0]):
    for c in range(hm_pivot.shape[1]):
        v = hm_pivot.values[r, c]
        if not np.isnan(v):
            ax.text(c, r, f"{v:.2f}", ha="center", va="center",
                    fontsize=8, color="black")
ax.set_title(f"Per-task Score Heatmap  (budget = {budget_sel:.0%} | baseline = full)")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot_per_task_heatmap.png", dpi=150)
plt.show()

# ── Plot 6: Per-task grouped bar at 20% budget ────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=False)
axes = axes.flatten()
for ax_i, task_name in enumerate(TASKS):
    ax = axes[ax_i]
    task_sub = summary_df[
        ((summary_df["kv_budget"] == budget_sel) |
         (summary_df["method"] == "baseline")) &
        (summary_df["task"] == task_name)
    ].copy()
    task_sub = task_sub.drop_duplicates(subset=["method"])
    col_map  = {m: METHOD_STYLES[m]["color"] for m in METHODS}
    x_labels = [m for m in METHODS if m in task_sub["method"].values]
    x_scores = [task_sub[task_sub["method"]==m]["score_mean"].values[0]
                for m in x_labels]
    colors   = [col_map[m] for m in x_labels]
    bars = ax.bar(range(len(x_labels)), x_scores, color=colors)
    ax.set_xticks(range(len(x_labels)))
    ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=7)
    ax.set_title(task_name, fontsize=9)
    ax.set_ylim(0, max(x_scores)*1.15 if x_scores else 1)
    ax.grid(axis="y", alpha=0.3)
    for bar, sc in zip(bars, x_scores):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f"{sc:.2f}", ha="center", va="bottom", fontsize=7)
fig.suptitle(f"Per-Task Scores — {budget_sel:.0%} KV Budget vs Baseline", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot_per_task_bars.png", dpi=150)
plt.show()

print(f"\nAll plots saved to {OUTPUT_DIR}/")